**Saved run (6 November 2026) with all outputs.** This is the exact run whose numbers are used in the report (versions, replication, STS sizes, bootstrap confidence intervals, box plot, error examples). It read the data from the author's Google Drive. The runnable version in the repository, `extra_analysis.ipynb`, is identical except that its first cell also accepts the `data/` folder of this repository.

# Extra analysis for the final report
COMP8240 — Trilok Kumar Reddy Ande

This notebook **only reads** the saved WikiSim files (it does not change them). It:
1. records the exact library versions,
2. re-runs the SimCSE replication with those versions,
3. counts the STS test-set sizes,
4. computes bootstrap 95% confidence intervals,
5. draws the WikiSim box plot,
6. lists example errors.

Run on a **T4 GPU** runtime: Runtime → Run all. When asked, allow access to Google Drive.

## 0. Data folder (read only)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/COMP8240_WikiSim"
import os
for f in ["wikisim_pairs.csv", "annotated_100.csv"]:
    print(f, "found" if os.path.exists(f"{SAVE_DIR}/{f}") else "MISSING")

Mounted at /content/drive
wikisim_pairs.csv found
annotated_100.csv found


## 1. Replication with recorded versions

In [ ]:
%cd /content
!rm -rf SimCSE
!git clone -q https://github.com/princeton-nlp/SimCSE.git
%cd /content/SimCSE
!pip install -q transformers scipy scikit-learn prettytable
!sed -i "s/tokenizer.batch_encode_plus(/tokenizer(/g" evaluation.py
%cd SentEval/data/downstream/
!bash download_dataset.sh > /dev/null 2>&1
%cd /content/SimCSE

/content
/content/SimCSE
/content/SimCSE/SentEval/data/downstream
/content/SimCSE


In [ ]:
import sys, torch, transformers, scipy, sklearn, pandas, numpy
print("Python      ", sys.version.split()[0])
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("scipy       ", scipy.__version__)
print("scikit-learn", sklearn.__version__)
print("pandas      ", pandas.__version__)
print("numpy       ", numpy.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

Python       3.13.16
torch        2.11.0+cu130
transformers 5.22.0
scipy        1.16.3
scikit-learn 1.6.1
pandas       2.2.3
numpy        2.4.6
GPU          Tesla T4


In [ ]:
!python evaluation.py --model_name_or_path princeton-nlp/sup-simcse-bert-base-uncased --pooler cls --task_set sts --mode test 2>&1 | tail -n 12

+-------+-------+-------+-------+-------+--------------+-----------------+-------+
| 75.30 | 84.67 | 80.19 | 85.40 | 80.82 |    84.26     |      80.39      | 81.58 |
+-------+-------+-------+-------+-------+--------------+-----------------+-------+
+------+------+------+------+------+------+------+------+
|  MR  |  CR  | SUBJ | MPQA | SST2 | TREC | MRPC | Avg. |
+------+------+------+------+------+------+------+------+
| 0.00 | 0.00 | 0.00 | 0.00 | 0.00 | 0.00 | 0.00 | 0.00 |
+------+------+------+------+------+------+------+------+
2026-11-06 02:13:03,228 : close.started
2026-11-06 02:13:03,229 : close.complete
2026-11-06 02:13:03,229 : close.started
2026-11-06 02:13:03,229 : close.complete


In [ ]:
!python evaluation.py --model_name_or_path princeton-nlp/unsup-simcse-bert-base-uncased --pooler cls_before_pooler --task_set sts --mode test 2>&1 | tail -n 12

+-------+-------+-------+-------+-------+--------------+-----------------+-------+
| 68.40 | 82.41 | 74.38 | 80.91 | 78.56 |    76.85     |      72.23      | 76.25 |
+-------+-------+-------+-------+-------+--------------+-----------------+-------+
+------+------+------+------+------+------+------+------+
|  MR  |  CR  | SUBJ | MPQA | SST2 | TREC | MRPC | Avg. |
+------+------+------+------+------+------+------+------+
| 0.00 | 0.00 | 0.00 | 0.00 | 0.00 | 0.00 | 0.00 | 0.00 |
+------+------+------+------+------+------+------+------+
2026-11-06 02:15:00,989 : close.started
2026-11-06 02:15:00,990 : close.complete
2026-11-06 02:15:00,990 : close.started
2026-11-06 02:15:00,990 : close.complete


## 2. STS test-set sizes

In [ ]:
import glob
base = "/content/SimCSE/SentEval/data/downstream"
for y in ["STS12", "STS13", "STS14", "STS15", "STS16"]:
    files = glob.glob(f"{base}/STS/{y}-en-test/STS.gs.*.txt")
    n = sum(sum(1 for l in open(f) if l.strip()) for f in files)
    print(y, n)
print("STS-B", sum(1 for l in open(f"{base}/STS/STSBenchmark/sts-test.csv") if l.strip()))
print("SICK-R", sum(1 for l in open(f"{base}/SICK/SICK_test_annotated.txt") if l.strip()) - 1)

STS12 6216
STS13 2250
STS14 3750
STS15 3000
STS16 1186
STS-B 1379
SICK-R 4927


## 3. Bootstrap 95% confidence intervals

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import spearmanr
rng = np.random.default_rng(42)

def boot(x, y, B=2000):
    x, y = np.asarray(x, float), np.asarray(y, float)
    n = len(x)
    vals = []
    for _ in range(B):
        i = rng.integers(0, n, n)
        vals.append(spearmanr(x[i], y[i]).correlation)
    return np.percentile(vals, [2.5, 97.5]) * 100

d400 = pd.read_csv(f"{SAVE_DIR}/wikisim_pairs.csv")
d100 = pd.read_csv(f"{SAVE_DIR}/annotated_100.csv")
print("Rows:", len(d400), "and", len(d100))

print("\n400 pairs, Spearman vs levels [95% CI]:")
for c in ["word_overlap", "unsup_simcse", "sup_simcse"]:
    lo, hi = boot(d400.level, d400[c])
    print(f"  {c:14s} {spearmanr(d400.level, d400[c]).correlation*100:.2f}  [{lo:.1f}, {hi:.1f}]")

print("\n100 pairs, Spearman vs author's scores [95% CI]:")
cols = ["word_overlap", "unsup_simcse", "sup_simcse"] + [c for c in ["llm_score_v2"] if c in d100.columns]
for c in cols:
    d = d100.dropna(subset=[c])
    lo, hi = boot(d.human_score, d[c])
    print(f"  {c:14s} {spearmanr(d.human_score, d[c]).correlation*100:.2f}  [{lo:.1f}, {hi:.1f}]")

# Is unsupervised really better than supervised on the 100 pairs? Paired bootstrap.
x, u, s = d100.human_score.values, d100.unsup_simcse.values, d100.sup_simcse.values
diffs = []
for _ in range(2000):
    i = rng.integers(0, len(x), len(x))
    diffs.append(spearmanr(x[i], u[i]).correlation - spearmanr(x[i], s[i]).correlation)
lo, hi = np.percentile(diffs, [2.5, 97.5]) * 100
print(f"\nUnsup minus Sup (100 pairs): {(spearmanr(x,u).correlation-spearmanr(x,s).correlation)*100:.2f}  95% CI [{lo:.1f}, {hi:.1f}]")
print("If the interval contains 0, the difference is not significant.")

Rows: 400 and 100

400 pairs, Spearman vs levels [95% CI]:
  word_overlap   30.29  [21.2, 38.8]
  unsup_simcse   70.13  [64.3, 74.9]
  sup_simcse     71.49  [65.9, 76.3]

100 pairs, Spearman vs author's scores [95% CI]:
  word_overlap   34.04  [14.2, 52.5]
  unsup_simcse   67.52  [56.2, 75.8]
  sup_simcse     67.31  [55.1, 76.9]
  llm_score_v2   62.50  [51.2, 72.7]

Unsup minus Sup (100 pairs): 0.21  95% CI [-8.0, 8.7]
If the interval contains 0, the difference is not significant.


## 4. Box plot for the report

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.6), sharey=True)
for ax, col, title in zip(axes, ["unsup_simcse", "sup_simcse"], ["Unsup-SimCSE", "Sup-SimCSE"]):
    ax.boxplot([d400[d400.level == l][col] for l in [0, 1, 2, 3]])
    ax.set_xticks([1, 2, 3, 4], ["L0", "L1", "L2", "L3"])
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("WikiSim level", fontsize=8)
axes[0].set_ylabel("Cosine similarity", fontsize=8)
plt.tight_layout()
plt.savefig("/content/wikisim_boxplot.pdf")
plt.savefig(f"{SAVE_DIR}/wikisim_boxplot.pdf")
plt.show()
print("Saved wikisim_boxplot.pdf to the Drive folder")

<Figure size 650x260 with 2 Axes>  (image not stored in this file; the same box plot is Figure 1 in Appendix B of the report)

Saved wikisim_boxplot.pdf to the Drive folder


## 5. Example errors

In [ ]:
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 250)
print("Level 1 pairs (different articles) that Sup-SimCSE scored HIGHEST:\n")
for _, r in d400[d400.level == 1].nlargest(3, "sup_simcse").iterrows():
    print(f"{r.sup_simcse:.3f} | {r.article1} / {r.article2}\n  1: {r.s1}\n  2: {r.s2}\n")
print("Level 2 pairs (same article) that Sup-SimCSE scored LOWEST:\n")
for _, r in d400[d400.level == 2].nsmallest(3, "sup_simcse").iterrows():
    print(f"{r.sup_simcse:.3f} | {r.article1} / {r.article2}\n  1: {r.s1}\n  2: {r.s2}\n")

Level 1 pairs (different articles) that Sup-SimCSE scored HIGHEST:

0.623 | Malaria / Measles
  1: By 2014, coverage of preventive malaria treatment during pregnancy had increased, with more than half of eligible women receiving at least one dose.
  2: A vaccination campaign brought the measles vaccination rate from 31 to 34% in 2018 to an estimated 94% of the eligible population in December 2019.

0.581 | Tuberculosis / Influenza
  1: Tuberculosis caused widespread public concern in the 19th and early 20th centuries as the disease became common among the urban poor.
  2: Subtypes of influenza A virus were discovered throughout the 1930s, and influenza B virus was discovered in 1940.

0.549 | Mars / Venus
  1: Deimos, being only just outside synchronous orbit – where the orbital period would match the planet's period of rotation – rises as expected in the east, but slowly.
  2: Venus can thus be north of the sun and appear as a morning star and an evening star on the same day, in the n

## 6. Exact versions for requirements.txt

In [ ]:
!pip freeze | grep -iE "^(transformers|torch|scipy|scikit-learn|prettytable|datasets|pandas|numpy|nltk|requests|accelerate|matplotlib)=="

accelerate==1.15.0
datasets==4.8.5
matplotlib==3.10.0
nltk==3.9.1
numpy==2.4.6
pandas==2.2.3
prettytable==3.18.0
requests==2.32.4
scikit-learn==1.6.1
scipy==1.16.3
torch==2.11.0+cu130
transformers==5.22.0
